# Gold Layer — Schedule, Cost & Portfolio KPIs
Produces 6 gold tables: project summary, cost analysis, subcontractor performance, weekly trends, overrun alerts, and portfolio scorecards.

In [ ]:
from pyspark.sql.functions import (
    col, count, sum as spark_sum, avg, round as spark_round,
    countDistinct, when, lit, weekofyear, year, max as spark_max,
    min as spark_min, current_timestamp
)
import pyspark.sql.functions as F

df_proj  = spark.read.format('delta').table('silver_projects')
df_tasks = spark.read.format('delta').table('silver_tasks')
df_costs = spark.read.format('delta').table('silver_cost_ledger')
df_subs  = spark.read.format('delta').table('silver_subcontractors')

In [ ]:
# Gold 1 — Project summary (status, schedule, cost)
project_summary = (
    df_proj
    .join(
        df_tasks.groupBy('project_id').agg(
            count('task_id').alias('total_tasks'),
            spark_sum('is_completed').alias('completed_tasks'),
            spark_sum('is_delayed').alias('delayed_tasks'),
            spark_round(avg('pct_complete'), 2).alias('avg_pct_complete'),
        ),
        'project_id', 'left'
    )
    .withColumn('task_completion_rate',
        spark_round(col('completed_tasks') / col('total_tasks') * 100, 2)
    )
    .withColumn('gold_timestamp', current_timestamp())
)
project_summary.write.mode('overwrite').format('delta').saveAsTable('gold_project_summary')
print(f'Gold project summary: {spark.table("gold_project_summary").count()} rows')

In [ ]:
# Gold 2 — Cost analysis by project and category
cost_analysis = (
    df_costs
    .groupBy('project_id', 'cost_category')
    .agg(
        count('cost_id').alias('entry_count'),
        spark_round(spark_sum('planned_cost'), 2).alias('total_planned'),
        spark_round(spark_sum('actual_cost'), 2).alias('total_actual'),
        spark_round(spark_sum('cost_variance'), 2).alias('total_variance'),
        spark_round(avg('cost_variance_pct'), 2).alias('avg_variance_pct'),
        spark_sum('is_overrun').alias('overrun_entries'),
    )
    .withColumn('overrun_rate',
        spark_round(col('overrun_entries') / col('entry_count') * 100, 2)
    )
    .withColumn('gold_timestamp', current_timestamp())
)
cost_analysis.write.mode('overwrite').format('delta').saveAsTable('gold_cost_analysis')
print(f'Gold cost analysis: {spark.table("gold_cost_analysis").count()} rows')

In [ ]:
# Gold 3 — Subcontractor performance
sub_perf = (
    df_tasks
    .join(df_subs.select('subcontractor_id', 'company_name', 'trade', 'rating', 'rating_band'), 'subcontractor_id', 'left')
    .groupBy('subcontractor_id', 'company_name', 'trade', 'rating', 'rating_band')
    .agg(
        count('task_id').alias('total_tasks'),
        spark_sum('is_completed').alias('completed_tasks'),
        spark_sum('is_delayed').alias('delayed_tasks'),
        spark_round(avg('schedule_variance_days'), 2).alias('avg_schedule_variance_days'),
        spark_round(avg('pct_complete'), 2).alias('avg_pct_complete'),
    )
    .withColumn('on_time_rate',
        spark_round((col('total_tasks') - col('delayed_tasks')) / col('total_tasks') * 100, 2)
    )
    .withColumn('gold_timestamp', current_timestamp())
)
sub_perf.write.mode('overwrite').format('delta').saveAsTable('gold_subcontractor_performance')
print(f'Gold subcontractor performance: {spark.table("gold_subcontractor_performance").count()} rows')

In [ ]:
# Gold 4 — Weekly cost trends
weekly_trends = (
    df_costs
    .withColumn('week', weekofyear('entry_date'))
    .withColumn('yr',   year('entry_date'))
    .groupBy('yr', 'week', 'cost_category')
    .agg(
        spark_round(spark_sum('planned_cost'), 2).alias('weekly_planned'),
        spark_round(spark_sum('actual_cost'), 2).alias('weekly_actual'),
        spark_round(spark_sum('cost_variance'), 2).alias('weekly_variance'),
        count('cost_id').alias('entry_count'),
    )
    .withColumn('gold_timestamp', current_timestamp())
)
weekly_trends.write.mode('overwrite').format('delta').saveAsTable('gold_weekly_cost_trends')
print(f'Gold weekly cost trends: {spark.table("gold_weekly_cost_trends").count()} rows')

In [ ]:
# Gold 5 — Overrun alerts (projects with critical schedule or cost risk)
overrun_alerts = (
    project_summary
    .filter(
        (col('schedule_risk_band').isin('Critical', 'High')) |
        (col('cost_risk_band').isin('Critical', 'High'))
    )
    .select(
        'project_id', 'project_name', 'project_type', 'region', 'status',
        'budget', 'schedule_variance_days', 'schedule_risk_band',
        'cost_variance_pct', 'cost_risk_band',
        'total_tasks', 'delayed_tasks', 'avg_pct_complete',
        'gold_timestamp'
    )
)
overrun_alerts.write.mode('overwrite').format('delta').saveAsTable('gold_overrun_alerts')
print(f'Gold overrun alerts: {spark.table("gold_overrun_alerts").count()} rows')

In [ ]:
# Gold 6 — Portfolio scorecards (one row per project, all KPIs)
portfolio_scorecards = (
    project_summary
    .join(
        df_costs.groupBy('project_id').agg(
            spark_round(spark_sum('planned_cost'), 2).alias('total_planned_cost'),
            spark_round(spark_sum('actual_cost'), 2).alias('total_actual_cost'),
            spark_round(spark_sum('cost_variance'), 2).alias('total_cost_variance'),
        ),
        'project_id', 'left'
    )
    .withColumn('performance_band',
        when(
            (col('schedule_risk_band').isin('On Track', 'Ahead')) &
            (col('cost_risk_band').isin('On Budget')), 'Excellent'
        )
        .when(
            (col('schedule_risk_band') == 'Medium') |
            (col('cost_risk_band') == 'Over Budget'), 'Good'
        )
        .when(
            (col('schedule_risk_band') == 'High') |
            (col('cost_risk_band') == 'High'), 'Needs Attention'
        )
        .otherwise('Critical')
    )
    .withColumn('gold_timestamp', current_timestamp())
)
portfolio_scorecards.write.mode('overwrite').format('delta').saveAsTable('gold_portfolio_scorecards')
print(f'Gold portfolio scorecards: {spark.table("gold_portfolio_scorecards").count()} rows')

In [ ]:
print('Gold aggregation complete.')
print(f'  Project summary         : {project_summary.count():>6,}')
print(f'  Cost analysis           : {cost_analysis.count():>6,}')
print(f'  Subcontractor perf.     : {sub_perf.count():>6,}')
print(f'  Weekly cost trends      : {weekly_trends.count():>6,}')
print(f'  Overrun alerts          : {overrun_alerts.count():>6,}')
print(f'  Portfolio scorecards    : {portfolio_scorecards.count():>6,}')